## Financial Feature Engineering

Three financial performance metrics were engineered from the original dataset:

- **Profit**, representing the monetary gain generated by each campaign after deducting marketing costs.
- **Return on Ad Spend (ROAS)**, measuring the revenue generated for every dollar invested in advertising.
- **Profit Margin**, expressing profitability as a percentage of campaign revenue.

These metrics provide a more comprehensive view of campaign performance than revenue or marketing spend alone and will support later analyses of profitability, budget allocation, and marketing efficiency.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/clean_campaign_data.csv", parse_dates=["startdate", "enddate"]
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   campaignid         10000 non-null  object        
 1   startdate          10000 non-null  datetime64[ns]
 2   enddate            10000 non-null  datetime64[ns]
 3   channel            10000 non-null  object        
 4   impressions        10000 non-null  int64         
 5   clicks             10000 non-null  int64         
 6   leads              10000 non-null  int64         
 7   conversions        10000 non-null  int64         
 8   cost_usd           10000 non-null  float64       
 9   revenue_usd        10000 non-null  float64       
 10  roi                10000 non-null  float64       
 11  campaign_duration  10000 non-null  int64         
dtypes: datetime64[ns](2), float64(3), int64(5), object(2)
memory usage: 937.6+ KB


In [5]:
df["profit"] = df["revenue_usd"] - df["cost_usd"]
df["roas"] = (
    df["revenue_usd"] /
    df["cost_usd"]
).round(2)
df["profit_margin"] = (
    (df["profit"] / df["revenue_usd"]) * 100
).round(2)

df[[
    "campaignid",
    "cost_usd",
    "revenue_usd",
    "profit",
    "roas",
    "profit_margin"
]].head()

,campaignid,cost_usd,revenue_usd,profit,roas,profit_margin
0,CAMP00001,1052.39,2236.02,1183.63,2.12,52.93
1,CAMP00002,3964.90,11740.15,7775.25,2.96,66.23
2,CAMP00003,1000.39,1902.24,901.85,1.90,47.41
3,CAMP00004,1252.63,2209.74,957.11,1.76,43.31
4,CAMP00005,4935.48,14111.31,9175.83,2.86,65.02


In [6]:
df[[
    "profit",
    "roas",
    "profit_margin"
]].describe()

,profit,roas,profit_margin
count,10000.000000,10000.000000,10000.000000
mean,2550.291939,2.001576,45.105733
std,2205.987067,0.578806,17.750340
min,0.690000,1.000000,0.030000
25%,738.992500,1.500000,33.320000
50%,1948.785000,1.990000,49.815000
75%,3833.025000,2.510000,60.090000
max,9903.910000,3.000000,66.660000


## Marketing Efficiency Feature Engineering

Six marketing efficiency metrics were engineered to evaluate campaign effectiveness across different stages of the customer acquisition funnel.

The engineered metrics include:

- Click-Through Rate (CTR)
- Lead Conversion Rate
- Customer Conversion Rate
- Cost per Click (CPC)
- Cost per Lead (CPL)
- Cost per Acquisition (CPA)

These KPIs provide a standardized framework for comparing campaign efficiency regardless of campaign size or marketing budget. They will serve as key performance indicators in the SQL analysis and Power BI dashboard.

In [8]:
df["ctr"] = (
    (df["clicks"] / df["impressions"]) * 100
).round(2)
df["lead_conversion_rate"] = (
    (df["leads"] / df["clicks"]) * 100
).round(2)
df["customer_conversion_rate"] = (
    (df["conversions"] / df["leads"]) * 100
).round(2)
df["cpc"] = (
    df["cost_usd"] / df["clicks"]
).round(2)
df["cpl"] = (
    df["cost_usd"] / df["leads"]
).round(2)
df["cpa"] = (
    df["cost_usd"] / df["conversions"]
).round(2)

df[[
    "campaignid",
    "ctr",
    "lead_conversion_rate",
    "customer_conversion_rate",
    "cpc",
    "cpl",
    "cpa"
]].head()

,campaignid,ctr,lead_conversion_rate,customer_conversion_rate,cpc,cpl,cpa
0,CAMP00001,7.95,49.90,46.29,0.05,0.09,0.20
1,CAMP00002,7.91,41.67,37.84,0.25,0.60,1.59
2,CAMP00003,6.88,48.81,42.24,0.06,0.12,0.29
3,CAMP00004,1.71,37.95,33.73,0.47,1.24,3.66
4,CAMP00005,1.46,36.61,37.15,1.19,3.24,8.74


In [9]:
df[[
    "ctr",
    "lead_conversion_rate",
    "customer_conversion_rate",
    "cpc",
    "cpl",
    "cpa"
]].describe().round(2)

,ctr,lead_conversion_rate,customer_conversion_rate,cpc,cpl,cpa
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,5.46,30.12,40.18,0.92,3.69,10.25
std,2.60,11.53,11.55,2.21,10.84,31.51
min,1.00,9.97,19.05,0.00,0.01,0.02
25%,3.24,20.16,30.21,0.16,0.52,1.32
50%,5.44,30.14,40.27,0.34,1.25,3.27
75%,7.71,40.17,50.28,0.82,3.15,8.45
max,10.00,49.99,59.99,57.10,490.18,1225.45


## Time-Based Feature Engineering

Additional temporal features were derived from the campaign start and end dates to support trend analysis and time-based reporting.

The engineered features include:

- Campaign Duration (days)
- Campaign Start Year
- Campaign Start Month
- Campaign Start Month Name
- Campaign Quarter
- Campaign Start Weekday

These variables simplify time-series analysis, seasonal performance comparisons, and dashboard filtering in later stages of the project.

In [11]:
df["campaign_duration"] = (
    df["enddate"] - df["startdate"]
).dt.days
df["start_year"] = df["startdate"].dt.year
df["start_month"] = df["startdate"].dt.month
df["start_month_name"] = df["startdate"].dt.month_name()
df["quarter"] = df["startdate"].dt.quarter
df["weekday"] = df["startdate"].dt.day_name()

df[[
    "campaignid",
    "startdate",
    "enddate",
    "campaign_duration",
    "start_year",
    "start_month",
    "start_month_name",
    "quarter",
    "weekday"
]].head()

,campaignid,startdate,enddate,campaign_duration,start_year,start_month,start_month_name,quarter,weekday
0,CAMP00001,2025-04-13,2025-04-19,6,2025,4,April,2,Sunday
1,CAMP00002,2025-12-15,2025-12-24,9,2025,12,December,4,Monday
2,CAMP00003,2025-09-28,2025-10-06,8,2025,9,September,3,Sunday
3,CAMP00004,2025-04-17,2025-04-30,13,2025,4,April,2,Thursday
4,CAMP00005,2025-03-13,2025-03-22,9,2025,3,March,1,Thursday


In [12]:
df["campaign_duration"].describe().round(2)
print(df["start_month_name"].value_counts().sort_index())
print(df["quarter"].value_counts().sort_index())
print(df["weekday"].value_counts())

start_month_name
April        865
August       857
December     840
February     759
January      859
July         838
June         841
March        824
May          864
November     838
October      837
September    778
Name: count, dtype: int64
quarter
1    2442
2    2570
3    2473
4    2515
Name: count, dtype: int64
weekday
Wednesday    1478
Monday       1462
Tuesday      1450
Sunday       1445
Thursday     1411
Saturday     1387
Friday       1367
Name: count, dtype: int64


In [13]:
df.info()
df.isnull().sum()
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   campaignid                10000 non-null  object        
 1   startdate                 10000 non-null  datetime64[ns]
 2   enddate                   10000 non-null  datetime64[ns]
 3   channel                   10000 non-null  object        
 4   impressions               10000 non-null  int64         
 5   clicks                    10000 non-null  int64         
 6   leads                     10000 non-null  int64         
 7   conversions               10000 non-null  int64         
 8   cost_usd                  10000 non-null  float64       
 9   revenue_usd               10000 non-null  float64       
 10  roi                       10000 non-null  float64       
 11  campaign_duration         10000 non-null  int64         
 12  profit             

,campaignid,startdate,enddate,channel,impressions,clicks,leads,conversions,cost_usd,revenue_usd,...,lead_conversion_rate,customer_conversion_rate,cpc,cpl,cpa,start_year,start_month,start_month_name,quarter,weekday
0,CAMP00001,2025-04-13,2025-04-19,Search,293520,23335,11643,5389,1052.39,2236.02,...,49.90,46.29,0.05,0.09,0.20,2025,4,April,2,Sunday
1,CAMP00002,2025-12-15,2025-12-24,Search,200340,15841,6601,2498,3964.90,11740.15,...,41.67,37.84,0.25,0.60,1.59,2025,12,December,4,Monday
2,CAMP00003,2025-09-28,2025-10-06,Email,239365,16478,8043,3397,1000.39,1902.24,...,48.81,42.24,0.06,0.12,0.29,2025,9,September,3,Sunday
3,CAMP00004,2025-04-17,2025-04-30,Search,156382,2672,1014,342,1252.63,2209.74,...,37.95,33.73,0.47,1.24,3.66,2025,4,April,2,Thursday
4,CAMP00005,2025-03-13,2025-03-22,Influencer,285472,4155,1521,565,4935.48,14111.31,...,36.61,37.15,1.19,3.24,8.74,2025,3,March,1,Thursday


In [18]:
import os

output_dir = "/content/processed"
os.makedirs(output_dir, exist_ok=True)
df.to_csv(os.path.join(output_dir, "marketing_campaign_engineered.csv"), index=False)

print("Engineered dataset exported successfully.")

Engineered dataset exported successfully.
